In [ ]:
# 1. Environment, imports, and deterministic seeds
import os, gc, json, random, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"
random.seed(SEED)

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from skimage.feature import hog
from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import tensorflow as tf
from tensorflow.keras import Model, layers, callbacks
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.applications.resnet50 import preprocess_input

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
try:
    tf.config.experimental.enable_op_determinism()
except Exception:
    pass

DATASET_PATH = Path("/kaggle/input/datasets/thilak02/breast-cancer-detection-using-thermography/BCD_Dataset")
OUTPUT_DIR = Path("/kaggle/working/final_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IMG_SIZE = 224
BATCH_SIZE = 16
CLASS_MAP = {"normal": 0, "Sick": 1}
print("TensorFlow:", tf.__version__)
print("Dataset:", DATASET_PATH)


In [ ]:
# 2. Load only the two paper classes; Unknown_class is intentionally excluded
def load_dataset(root):
    images, labels, filenames = [], [], []
    for class_name, label in CLASS_MAP.items():
        folder = root / class_name
        if not folder.exists():
            raise FileNotFoundError(f"Missing class folder: {folder}")
        paths = sorted(p for p in folder.rglob("*") if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"})
        for path in paths:
            bgr = cv2.imread(str(path))
            if bgr is None:
                print("Skipped unreadable:", path)
                continue
            rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            rgb = cv2.resize(rgb, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            images.append(rgb)
            labels.append(label)
            filenames.append(str(path))
    return np.asarray(images, dtype=np.uint8), np.asarray(labels, dtype=np.int64), np.asarray(filenames)

images, labels, filenames = load_dataset(DATASET_PATH)
assert len(images) == len(labels) == len(filenames) and len(images) > 0
print("Images:", images.shape)
print(pd.Series(labels).map({0: "Normal", 1: "Abnormal"}).value_counts())


In [ ]:
# 3. The single fixed 70/15/15 stratified split used by every experiment
indices = np.arange(len(images))
train_indices, temp_indices = train_test_split(
    indices, test_size=0.30, stratify=labels, random_state=SEED
)
val_indices, test_indices = train_test_split(
    temp_indices, test_size=0.50, stratify=labels[temp_indices], random_state=SEED
)

assert set(train_indices).isdisjoint(val_indices)
assert set(train_indices).isdisjoint(test_indices)
assert set(val_indices).isdisjoint(test_indices)
assert len(train_indices) + len(val_indices) + len(test_indices) == len(images)

X_train, y_train = images[train_indices], labels[train_indices]
X_val, y_val = images[val_indices], labels[val_indices]
X_test, y_test = images[test_indices], labels[test_indices]
train_filenames, val_filenames, test_filenames = (
    filenames[train_indices], filenames[val_indices], filenames[test_indices]
)

split_manifest = pd.DataFrame({
    "filename": np.concatenate([train_filenames, val_filenames, test_filenames]),
    "label": np.concatenate([y_train, y_val, y_test]),
    "split": (["train"] * len(y_train) + ["validation"] * len(y_val) + ["test"] * len(y_test))
})
split_manifest.to_csv(OUTPUT_DIR / "split_manifest.csv", index=False)
print(pd.crosstab(split_manifest["split"], split_manifest["label"], margins=True))


In [ ]:
# 4. Shared metrics registry: probabilities, not hard labels, are used for ROC-AUC
results = []
predictions = {}

def register_result(name, y_true, y_prob, threshold=0.5, evaluation="hold-out test"):
    y_prob = np.asarray(y_prob).reshape(-1)
    y_pred = (y_prob >= threshold).astype(int)
    row = {
        "Model": name,
        "Evaluation": evaluation,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall_Sensitivity": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
    }
    results.append(row)
    predictions[name] = {"true": np.asarray(y_true), "prob": y_prob, "pred": y_pred}
    print(pd.DataFrame([row]).round(4).to_string(index=False))
    return row

early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True, min_delta=1e-4
)


## Experiment 1 — ResNet50 baseline




In [ ]:
# 5. ResNet50 baseline
def make_resnet50():
    base = ResNet50(weights="imagenet", include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    x = layers.Lambda(preprocess_input)(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = Model(inp, out, name="ResNet50_baseline")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model

resnet_model = make_resnet50()
resnet_history = resnet_model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=30, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=1
)
resnet_model.save(OUTPUT_DIR / "resnet50_baseline.keras")
resnet_prob = resnet_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel()
register_result("ResNet50 baseline", y_test, resnet_prob)


## Experiments 2–3 — HOG + XGBoost, without and with augmentation



In [ ]:
# 6. HOG features and training-only augmentation
def extract_hog_batch(batch):
    feats = []
    for image in batch:
        gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
        feats.append(hog(gray, orientations=9, pixels_per_cell=(16, 16),
                          cells_per_block=(2, 2), block_norm="L2-Hys"))
    return np.asarray(feats, dtype=np.float32)

augmenter = tf.keras.Sequential([
    layers.RandomFlip("horizontal", seed=SEED),
    layers.RandomRotation(15 / 360, fill_mode="reflect", seed=SEED + 1),
    layers.RandomBrightness(0.10, value_range=(0.0, 1.0), seed=SEED + 2),
    layers.GaussianNoise(0.02, seed=SEED + 3),
])

X_train_aug_only = np.clip(
    augmenter(tf.cast(X_train, tf.float32) / 255.0, training=True).numpy() * 255.0,
    0, 255
).astype(np.uint8)
X_train_with_aug = np.concatenate([X_train, X_train_aug_only])
y_train_with_aug = np.concatenate([y_train, y_train])

H_train = extract_hog_batch(X_train)
H_val = extract_hog_batch(X_val)
H_test = extract_hog_batch(X_test)
H_train_aug = extract_hog_batch(X_train_with_aug)

def fit_hog_transform(train_features, *others):
    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_features)
    max_components = min(200, train_scaled.shape[0] - 1, train_scaled.shape[1])
    pca = PCA(n_components=max_components, random_state=SEED)
    transformed = [pca.fit_transform(train_scaled)]
    transformed.extend(pca.transform(scaler.transform(x)) for x in others)
    return scaler, pca, transformed

def make_xgb():
    return XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.03,
        subsample=0.85, colsample_bytree=0.85,
        objective="binary:logistic", eval_metric="logloss",
        random_state=SEED, n_jobs=-1
    )

scaler_noaug, pca_noaug, (Ht, Hv, Hs) = fit_hog_transform(H_train, H_val, H_test)
xgb_noaug = make_xgb()
xgb_noaug.fit(Ht, y_train)
register_result("HOG + XGBoost (no augmentation)", y_test, xgb_noaug.predict_proba(Hs)[:, 1])

scaler_aug, pca_aug, (Hta, Hva, Hsa) = fit_hog_transform(H_train_aug, H_val, H_test)
xgb_aug = make_xgb()
xgb_aug.fit(Hta, y_train_with_aug)
register_result("HOG + XGBoost (augmentation)", y_test, xgb_aug.predict_proba(Hsa)[:, 1])


## Experiment 4 — SVM kernel comparison




In [ ]:
# 7. Select SVM kernel using validation only
svm_validation = []
svm_candidates = {}
for kernel in ["linear", "rbf", "poly", "sigmoid"]:
    candidate = SVC(kernel=kernel, C=1.0, probability=True, class_weight="balanced", random_state=SEED)
    candidate.fit(Ht, y_train)
    val_prob = candidate.predict_proba(Hv)[:, 1]
    score = roc_auc_score(y_val, val_prob)
    svm_validation.append({"kernel": kernel, "validation_roc_auc": score})
    svm_candidates[kernel] = candidate

svm_validation = pd.DataFrame(svm_validation).sort_values("validation_roc_auc", ascending=False)
display(svm_validation)
best_kernel = svm_validation.iloc[0]["kernel"]
best_svm = svm_candidates[best_kernel]
register_result(f"HOG + SVM ({best_kernel}, validation-selected)", y_test,
                best_svm.predict_proba(Hs)[:, 1])


## Experiment 5 — CBAM/CNN features + HOG + LightGBM

In [ ]:
# 8. CBAM teacher and hybrid feature extractor
def cbam_block(x, ratio=8):
    channels = int(x.shape[-1])
    shared_1 = layers.Dense(max(channels // ratio, 1), activation="relu")
    shared_2 = layers.Dense(channels)
    avg = shared_2(shared_1(layers.GlobalAveragePooling2D()(x)))
    mx = shared_2(shared_1(layers.GlobalMaxPooling2D()(x)))
    channel = layers.Reshape((1, 1, channels))(layers.Activation("sigmoid")(layers.Add()([avg, mx])))
    x = layers.Multiply()([x, channel])
    avg_spatial = layers.Lambda(lambda z: tf.reduce_mean(z, axis=-1, keepdims=True))(x)
    max_spatial = layers.Lambda(lambda z: tf.reduce_max(z, axis=-1, keepdims=True))(x)
    spatial = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(
        layers.Concatenate()([avg_spatial, max_spatial])
    )
    return layers.Multiply()([x, spatial])

def make_cbam_teacher():
    base = EfficientNetB0(weights="imagenet", include_top=False,
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
    # Keras EfficientNet includes input rescaling and expects RGB values in [0,255].
    x = base(inp, training=False)
    x = cbam_block(x)
    embedding = layers.GlobalAveragePooling2D(name="cbam_embedding")(x)
    x = layers.Dropout(0.35)(embedding)
    out = layers.Dense(1, activation="sigmoid", name="teacher_probability")(x)
    model = Model(inp, out, name="CBAM_teacher")
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy", metrics=["accuracy", tf.keras.metrics.AUC(name="auc")])
    return model

cbam_teacher = make_cbam_teacher()
cbam_teacher.fit(X_train, y_train, validation_data=(X_val, y_val),
                 epochs=30, batch_size=BATCH_SIZE, callbacks=[early_stop], verbose=1)
cbam_teacher.save(OUTPUT_DIR / "cbam_teacher.keras")
embedding_model = Model(cbam_teacher.input, cbam_teacher.get_layer("cbam_embedding").output)

E_train = embedding_model.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
E_val = embedding_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
E_test = embedding_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

# Reuse HOG PCA learned from training only, then scale the concatenated representation.
F_train = np.hstack([Ht, E_train])
F_val = np.hstack([Hv, E_val])
F_test = np.hstack([Hs, E_test])
fusion_scaler = StandardScaler()
F_train_s = fusion_scaler.fit_transform(F_train)
F_val_s = fusion_scaler.transform(F_val)
F_test_s = fusion_scaler.transform(F_test)

lgb_hybrid = LGBMClassifier(
    n_estimators=300, learning_rate=0.03, num_leaves=15,
    subsample=0.85, colsample_bytree=0.85, class_weight="balanced",
    random_state=SEED, verbosity=-1
)
lgb_hybrid.fit(F_train_s, y_train)
register_result("CBAM features + HOG + LightGBM", y_test,
                lgb_hybrid.predict_proba(F_test_s)[:, 1])


## Experiment 6 — Teacher-assisted LightGBM




In [ ]:
# 9. Teacher-assisted LightGBM
teacher_train = cbam_teacher.predict(X_train, batch_size=BATCH_SIZE, verbose=0)
teacher_val = cbam_teacher.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
teacher_test = cbam_teacher.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
T_train = np.hstack([Ht, teacher_train])
T_val = np.hstack([Hv, teacher_val])
T_test = np.hstack([Hs, teacher_test])

teacher_student = LGBMClassifier(
    n_estimators=300, learning_rate=0.03, num_leaves=15,
    class_weight="balanced", random_state=SEED, verbosity=-1
)
teacher_student.fit(T_train, y_train)
register_result("Teacher-assisted HOG + LightGBM", y_test,
                teacher_student.predict_proba(T_test)[:, 1])


## Experiment 7 — Out-of-fold stacking




In [ ]:
# 10. Leakage-free OOF stacking on fused features
base_templates = [
    LGBMClassifier(n_estimators=250, learning_rate=0.03, num_leaves=15,
                   class_weight="balanced", random_state=SEED, verbosity=-1),
    SVC(kernel=best_kernel, C=1.0, probability=True, class_weight="balanced", random_state=SEED),
    RandomForestClassifier(n_estimators=400, class_weight="balanced",
                           random_state=SEED, n_jobs=-1),
    LogisticRegression(max_iter=3000, class_weight="balanced", random_state=SEED),
]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof = np.zeros((len(y_train), len(base_templates)))
test_meta = np.zeros((len(y_test), len(base_templates)))
fitted_base_models = []

for m, template in enumerate(base_templates):
    fold_test, models_for_type = [], []
    for fit_idx, hold_idx in skf.split(F_train_s, y_train):
        fold_model = clone(template)
        fold_model.fit(F_train_s[fit_idx], y_train[fit_idx])
        oof[hold_idx, m] = fold_model.predict_proba(F_train_s[hold_idx])[:, 1]
        fold_test.append(fold_model.predict_proba(F_test_s)[:, 1])
        models_for_type.append(fold_model)
    test_meta[:, m] = np.mean(fold_test, axis=0)
    fitted_base_models.append(models_for_type)

meta_model = LogisticRegression(class_weight="balanced", random_state=SEED)
meta_model.fit(oof, y_train)
stack_prob = meta_model.predict_proba(test_meta)[:, 1]
register_result("OOF stacking ensemble", y_test, stack_prob)


## Experiment 8 — Few-shot prototype classifier

In [ ]:
# 11. Few-shot CNN + prototype ensemble using frozen ResNet embeddings
proto_backbone = ResNet50(weights="imagenet", include_top=False, pooling="avg",
                          input_shape=(IMG_SIZE, IMG_SIZE, 3))
def resnet_embeddings(x):
    return proto_backbone.predict(preprocess_input(x.astype(np.float32)),
                                  batch_size=BATCH_SIZE, verbose=0)

P_train = resnet_embeddings(X_train)
P_test = resnet_embeddings(X_test)
prototypes = np.vstack([P_train[y_train == c].mean(axis=0) for c in [0, 1]])
d0 = np.linalg.norm(P_test - prototypes[0], axis=1)
d1 = np.linalg.norm(P_test - prototypes[1], axis=1)
proto_prob = 1.0 / (1.0 + np.exp(np.clip(d1 - d0, -50, 50)))
few_shot_ensemble_prob = 0.5 * proto_prob + 0.5 * resnet_prob
register_result("Few-shot CNN + prototype ensemble", y_test, few_shot_ensemble_prob)


## Experiment 9 — Snapshot ensemble




In [ ]:
# 12. Snapshot ensemble (three short cosine cycles)
snapshot_model = make_resnet50()
snapshot_weights = []
cycles, epochs_per_cycle = 3, 4
for cycle in range(cycles):
    lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=1e-3, decay_steps=max(1, epochs_per_cycle * int(np.ceil(len(y_train) / BATCH_SIZE)))
    )
    snapshot_model.compile(optimizer=tf.keras.optimizers.Adam(lr_schedule),
                           loss="binary_crossentropy", metrics=["accuracy"])
    snapshot_model.fit(X_train, y_train, validation_data=(X_val, y_val),
                       epochs=epochs_per_cycle, batch_size=BATCH_SIZE, verbose=1)
    path = OUTPUT_DIR / f"snapshot_{cycle + 1}.weights.h5"
    snapshot_model.save_weights(path)
    snapshot_weights.append(path)

snapshot_probs = []
for path in snapshot_weights:
    snapshot_model.load_weights(path)
    snapshot_probs.append(snapshot_model.predict(X_test, batch_size=BATCH_SIZE, verbose=0).ravel())
register_result("Snapshot ensemble", y_test, np.mean(snapshot_probs, axis=0))


## Experiment 10 — Five-fold cross-validation




In [ ]:
# 13. Five-fold CV on development data only
H_dev = np.vstack([H_train, H_val])
y_dev = np.concatenate([y_train, y_val])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for fold, (fit_idx, hold_idx) in enumerate(cv.split(H_dev, y_dev), start=1):
    scaler = StandardScaler()
    X_fit = scaler.fit_transform(H_dev[fit_idx])
    X_hold = scaler.transform(H_dev[hold_idx])
    n_comp = min(200, X_fit.shape[0] - 1, X_fit.shape[1])
    pca = PCA(n_components=n_comp, random_state=SEED)
    X_fit = pca.fit_transform(X_fit)
    X_hold = pca.transform(X_hold)
    model = make_xgb()
    model.fit(X_fit, y_dev[fit_idx])
    prob = model.predict_proba(X_hold)[:, 1]
    pred = (prob >= 0.5).astype(int)
    cv_rows.append({
        "fold": fold,
        "accuracy": accuracy_score(y_dev[hold_idx], pred),
        "f1": f1_score(y_dev[hold_idx], pred),
        "roc_auc": roc_auc_score(y_dev[hold_idx], prob),
    })
cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(OUTPUT_DIR / "five_fold_cv.csv", index=False)
display(cv_results.round(4))
print("Mean ± SD")
for col in ["accuracy", "f1", "roc_auc"]:
    print(f"{col}: {cv_results[col].mean():.4f} ± {cv_results[col].std(ddof=1):.4f}")


## Final paper-ready outputs




In [ ]:
# 14. Final table, ROC curves, confusion matrices, and training plot
results_df = pd.DataFrame(results).sort_values("ROC_AUC", ascending=False).reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / "final_holdout_results.csv", index=False)
display(results_df.round(4))

plt.figure(figsize=(9, 7))
for name, item in predictions.items():
    fpr, tpr, _ = roc_curve(item["true"], item["prob"])
    auc = roc_auc_score(item["true"], item["prob"])
    plt.plot(fpr, tpr, linewidth=1.8, label=f"{name} ({auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Hold-out test ROC curves")
plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curves.png", dpi=300)
plt.show()

best_name = results_df.iloc[0]["Model"]
best = predictions[best_name]
cm = confusion_matrix(best["true"], best["pred"])
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.colorbar()
tick_labels = ["Normal", "Abnormal"]
plt.xticks([0, 1], tick_labels)
plt.yticks([0, 1], tick_labels)
threshold = cm.max() / 2.0
for row in range(cm.shape[0]):
    for col in range(cm.shape[1]):
        plt.text(
            col, row, str(cm[row, col]),
            ha="center", va="center",
            color="white" if cm[row, col] > threshold else "black"
        )
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "best_confusion_matrix.png", dpi=300)
plt.show()

with open(OUTPUT_DIR / "run_metadata.json", "w") as f:
    json.dump({
        "seed": SEED, "image_size": IMG_SIZE, "split": "70/15/15 stratified",
        "n_total": int(len(labels)), "n_train": int(len(y_train)),
        "n_validation": int(len(y_val)), "n_test": int(len(y_test)),
        "best_model_by_test_roc_auc": best_name
    }, f, indent=2)
print("Saved outputs to:", OUTPUT_DIR)


## Complete per-model evaluation outputs

The following cells create combined and individual confusion matrices and ROC curves for every completed hold-out model. The final cells perform leakage-safe five-fold validation for HOG–XGBoost and RBF-SVM. Deep and ensemble models must be rebuilt inside every fold; reusing the already trained deep embeddings would leak fold information.


In [ ]:
# 15. Combined confusion matrices for every hold-out model
import math
import re

model_names = list(predictions.keys())
n_cols = 3
n_rows = math.ceil(len(model_names) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4.5 * n_rows))
axes = np.asarray(axes).reshape(-1)

for ax, model_name in zip(axes, model_names):
    values = predictions[model_name]
    cm = confusion_matrix(values["true"], values["pred"], labels=[0, 1])
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    threshold = cm.max() / 2.0
    for row in range(2):
        for col in range(2):
            ax.text(col, row, str(cm[row, col]), ha="center", va="center",
                    fontsize=13, fontweight="bold",
                    color="white" if cm[row, col] > threshold else "black")
    ax.set_title(model_name, fontsize=10)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_xticks([0, 1], ["Normal", "Abnormal"])
    ax.set_yticks([0, 1], ["Normal", "Abnormal"])

for ax in axes[len(model_names):]:
    ax.axis("off")

fig.colorbar(image, ax=axes.tolist(), fraction=0.015, pad=0.02)
fig.suptitle("Confusion Matrices for All Hold-Out Models",
             fontsize=16, fontweight="bold")
plt.subplots_adjust(top=0.93, hspace=0.45, wspace=0.30)
plt.savefig(OUTPUT_DIR / "all_confusion_matrices.png",
            dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 16. Save an individual confusion matrix for every model
for model_name, values in predictions.items():
    cm = confusion_matrix(values["true"], values["pred"], labels=[0, 1])
    fig, ax = plt.subplots(figsize=(5, 4))
    image = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(image, ax=ax)
    threshold = cm.max() / 2.0
    for row in range(2):
        for col in range(2):
            ax.text(col, row, str(cm[row, col]), ha="center", va="center",
                    fontsize=14, fontweight="bold",
                    color="white" if cm[row, col] > threshold else "black")
    ax.set_xticks([0, 1], ["Normal", "Abnormal"])
    ax.set_yticks([0, 1], ["Normal", "Abnormal"])
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    ax.set_title(model_name)
    plt.tight_layout()
    safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", model_name).strip("_").lower()
    plt.savefig(OUTPUT_DIR / f"confusion_matrix_{safe_name}.png",
                dpi=300, bbox_inches="tight")
    plt.close(fig)

print(f"Saved {len(predictions)} individual confusion matrices.")


In [ ]:
# 17. Combined ROC-AUC curves for every hold-out model
plt.figure(figsize=(11, 8))
colors = plt.cm.tab10(np.linspace(0, 1, len(predictions)))
auc_rows = []

for color, (model_name, values) in zip(colors, predictions.items()):
    fpr, tpr, _ = roc_curve(values["true"], values["prob"])
    auc_value = roc_auc_score(values["true"], values["prob"])
    auc_rows.append({"Model": model_name, "ROC_AUC": auc_value})
    plt.plot(fpr, tpr, color=color, linewidth=2,
             label=f"{model_name} (AUC={auc_value:.3f})")

plt.plot([0, 1], [0, 1], "k--", linewidth=1.5, label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves for All Hold-Out Models", fontweight="bold")
plt.grid(alpha=0.25)
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "all_models_roc_curves.png",
            dpi=300, bbox_inches="tight")
plt.show()

auc_summary_df = (
    pd.DataFrame(auc_rows)
    .sort_values("ROC_AUC", ascending=False)
    .reset_index(drop=True)
)
auc_summary_df.to_csv(OUTPUT_DIR / "all_models_auc.csv", index=False)
display(auc_summary_df.round(4))


In [ ]:
# 18. Save an individual ROC curve for every model
for model_name, values in predictions.items():
    fpr, tpr, _ = roc_curve(values["true"], values["prob"])
    auc_value = roc_auc_score(values["true"], values["prob"])
    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, linewidth=2.5, label=f"ROC-AUC = {auc_value:.4f}")
    plt.plot([0, 1], [0, 1], "k--", label="Random classifier")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(model_name)
    plt.grid(alpha=0.25)
    plt.legend(loc="lower right")
    plt.tight_layout()
    safe_name = re.sub(r"[^a-zA-Z0-9]+", "_", model_name).strip("_").lower()
    plt.savefig(OUTPUT_DIR / f"roc_curve_{safe_name}.png",
                dpi=300, bbox_inches="tight")
    plt.close()

print(f"Saved {len(predictions)} individual ROC curves.")


In [ ]:
# 19. Five-fold validation for HOG-XGBoost and RBF-SVM
H_all = extract_hog_batch(images)
all_model_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_all_rows = []

def append_cv_metrics(model_name, fold, y_true, y_prob):
    y_pred = (np.asarray(y_prob) >= 0.5).astype(int)
    cv_all_rows.append({
        "Model": model_name,
        "Fold": fold,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Sensitivity": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Specificity": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_prob),
    })

for fold, (fit_idx, hold_idx) in enumerate(
        all_model_cv.split(H_all, labels), start=1):
    print(f"Running fold {fold}/5")
    H_fit, H_hold = H_all[fit_idx], H_all[hold_idx]
    y_fit, y_hold = labels[fit_idx], labels[hold_idx]

    fold_scaler = StandardScaler()
    H_fit_scaled = fold_scaler.fit_transform(H_fit)
    H_hold_scaled = fold_scaler.transform(H_hold)

    n_components = min(
        200, H_fit_scaled.shape[0] - 1, H_fit_scaled.shape[1]
    )
    fold_pca = PCA(n_components=n_components, random_state=SEED)
    H_fit_pca = fold_pca.fit_transform(H_fit_scaled)
    H_hold_pca = fold_pca.transform(H_hold_scaled)

    fold_xgb = make_xgb()
    fold_xgb.fit(H_fit_pca, y_fit)
    append_cv_metrics(
        "HOG + XGBoost", fold, y_hold,
        fold_xgb.predict_proba(H_hold_pca)[:, 1]
    )

    fold_svm = SVC(
        kernel="rbf", C=1.0, probability=True,
        class_weight="balanced", random_state=SEED
    )
    fold_svm.fit(H_fit_pca, y_fit)
    append_cv_metrics(
        "HOG + SVM (RBF)", fold, y_hold,
        fold_svm.predict_proba(H_hold_pca)[:, 1]
    )

all_models_cv_df = pd.DataFrame(cv_all_rows)
all_models_cv_df.to_csv(
    OUTPUT_DIR / "five_fold_classical_models.csv", index=False
)
display(all_models_cv_df.round(4))


In [ ]:
# 20. Five-fold mean ± standard deviation table and graphs
metric_columns = [
    "Accuracy", "Precision", "Sensitivity",
    "Specificity", "F1", "ROC_AUC"
]

mean_df = all_models_cv_df.groupby("Model")[metric_columns].mean()
std_df = all_models_cv_df.groupby("Model")[metric_columns].std(ddof=1)

summary_rows = []
for model_name in mean_df.index:
    row = {"Model": model_name}
    for metric in metric_columns:
        row[metric] = (
            f"{mean_df.loc[model_name, metric]:.4f} ± "
            f"{std_df.loc[model_name, metric]:.4f}"
        )
    summary_rows.append(row)

five_fold_summary_df = pd.DataFrame(summary_rows)
five_fold_summary_df.to_csv(
    OUTPUT_DIR / "five_fold_all_metrics_summary.csv", index=False
)
display(five_fold_summary_df)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, metric in zip(axes, ["Accuracy", "F1", "ROC_AUC"]):
    for model_name, group in all_models_cv_df.groupby("Model"):
        ax.plot(group["Fold"], group[metric], marker="o", label=model_name)
    ax.set_title(f"Five-Fold {metric}")
    ax.set_xlabel("Fold")
    ax.set_ylabel(metric)
    ax.set_xticks(range(1, 6))
    ax.grid(alpha=0.25)

axes[-1].legend(fontsize=8, loc="best")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "five_fold_model_comparison.png",
            dpi=300, bbox_inches="tight")
plt.show()

print("All evaluation outputs saved to:", OUTPUT_DIR)
